# Generate Kaggle Competition Datasets

## Task
Participants receive a set of product pairs and must predict which product in each pair was rated as more similar (mean similarity rating) to its animal-based counterpart by human panelists — without access to the ratings themselves.

## Method

**Products dataset** (`products.csv`):
- Includes *all* products — animal-based, vegetarian, and plant-based — from both the Nectar sensory panel studies and the Taste Like ingredient database.
- Each product has an ingredient list and nutritional information. No brand names.
- Product codes are remapped to new anonymized IDs (reproducible via independent per-step RNGs, all seeded at 42).

**Taste Like subsetting** (1,726 raw → ~217 final):
- The full Taste Like database has 1,726 product variants across meat, dairy, seafood, eggs, and pet food. Of these, 1,036 map to Nectar's 25 meat/dairy subcategories (seafood, eggs, pet food, and unmapped products are excluded).
- **Deduplication**: Products with ≥ 80% Jaccard ingredient similarity to any Nectar product in the same category are removed (~40 products), preventing answer leakage and detectable duplicates.
- **Capping**: Taste Like products are capped at 1× the Nectar ranking count per category to keep pair counts tractable while maintaining ~50/50 anonymity per category.
- **Indistinguishability**: Missing nutrition columns are imputed from per-category Nectar distributions, and NaN rates are matched to Nectar's per-category patterns so Taste Like products cannot be identified by their data profile.

**Ranking pairs** (`ranking_pairs.csv`):
- Pairs are generated only for the *alternative protein* subset: meat-free products in meat analog categories and dairy-free products in dairy analog categories.
- All pairwise combinations within each category; 50% of pairs are randomly swapped to prevent ordering bias.

**Ground truth** (`solution.csv`):
- For each pair where *both* products are Nectar products from 2025 or 2026 with similarity scores, the label is the product code of the higher-rated product.
- All other pairs (involving Taste Like products, pre-2025 Nectar products, or products without similarity data) are labeled -1 (unscored).
- A `Usage` column marks each pair as `Public` or `Private` for Kaggle leaderboard evaluation.
- Scored pairs are randomly split 50/50 stratified by category. Unscored pairs are assigned to Private.

**Dropped categories**: Cold_Unbreaded_Chicken_Breast (only 1 ranking product, 0 scored pairs) and Tenders (no Nectar products, 0 scored pairs).

## Outputs
| File | Description |
|---|---|
| `products.csv` | All products with ingredients & nutrition |
| `ranking_pairs.csv` | Pairs for participants to rank |
| `solution.csv` | Ground truth labels + Public/Private usage |
| `sample_submission.csv` | Submission template |
| `product_code_map.csv` | Internal: original ↔ new code mapping |

In [1]:
import pandas as pd
import numpy as np
from itertools import combinations
from pathlib import Path

# Jupyter sets cwd to the notebook's directory
_notebook_dir = Path(".").resolve()
base_dir = _notebook_dir.parent.parent
output_dir = _notebook_dir / "dataset"
output_dir.mkdir(parents=True, exist_ok=True)

In [2]:
from taste_like_category_map import PARENT_SLUG_TO_NECTAR_CATEGORY

NUTRITION_COLS = [
    "Serving Size (g)",
    "Calories",
    "Total Fat (g)",
    "Saturated Fat (g)",
    "Trans Fat (g)",
    "Polyunsaturated Fat (g)",
    "Monounsaturated Fat (g)",
    "Cholesterol (mg)",
    "Sodium (mg)",
    "Total Carbohydrate (g)",
    "Dietary Fiber (g)",
    "Total Sugars (g)",
    "Added Sugars (g)",
    "Protein (g)",
    "Vitamin D (mcg)",
    "Calcium (mg)",
    "Iron (mg)",
    "Potassium (mg)",
]

IMPUTE_COLS = [
    "Saturated Fat (g)",
    "Trans Fat (g)",
    "Polyunsaturated Fat (g)",
    "Monounsaturated Fat (g)",
    "Dietary Fiber (g)",
    "Total Sugars (g)",
    "Added Sugars (g)",
    "Vitamin D (mcg)",
    "Potassium (mg)",
]

OUTPUT_COLS = [
    "Product code",
    "Category",
    "Ingredient List",
] + NUTRITION_COLS

In [ ]:
# Cold_Unbreaded_Chicken_Breast has 2 NECTAR products (1 pair)
# Tenders has no NECTAR products with a similarity rating
DROP_CATEGORIES = ["Cold_Unbreaded_Chicken_Breast", "Tenders"]

DAIRY_CATEGORIES = {
    "Barista_Milk",
    "Butter",
    "Cheddar_Cheese",
    "Cream_Cheese",
    "Creamer",
    "Ice_Cream_Hard_Serve",
    "Milk",
    "Mozzarella",
    "Sour_Cream",
    "Yogurt",
}
MEAT_CATEGORIES = {
    "Bacon",
    "Bratwurst",
    "Breaded_Chicken_Filet",
    "Breakfast_Sausages",
    "Burgers",
    "Chicken_Strips",
    "Deli_Ham",
    "Deli_Turkey",
    "Hot_Dogs",
    "Meatballs",
    "Nuggets",
    "Pulled_Pork",
    "Steak",
    "Unbreaded_Chicken_Breast",
}


def _add_product_type_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Derive product_type, is_meat_category, is_dairy_category from Category."""
    df["is_meat_category"] = df["Category"].isin(MEAT_CATEGORIES)
    df["is_dairy_category"] = df["Category"].isin(DAIRY_CATEGORIES)
    # All products in this dataset are alternative proteins:
    # meat-free in meat categories, dairy-free in dairy categories
    df["product_type"] = df["Category"].apply(
        lambda c: "meat-free" if c in MEAT_CATEGORIES else "dairy-free"
    )
    return df


def load_nectar_data(path: str, sensory_path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.dropna(subset=["Product_Code"])
    df["Product_Code"] = df["Product_Code"].astype(int)
    df = df[~df["Category"].isin(DROP_CATEGORIES)].copy()
    df["source"] = "nectar"
    df.rename(
        columns={"Manually Cleaned Ingredients": "Clean Ingredient List"},
        inplace=True,
    )
    # Keep only products that have similarity ratings
    sensory = pd.read_csv(sensory_path)
    scored = sensory[sensory["similarity"].notna()]
    scored_keys = set(zip(scored["product_category"], scored["product_code"]))
    df = df[
        df.apply(lambda r: (r["Category"], r["Product_Code"]) in scored_keys, axis=1)
    ].copy()

    # Cap unreasonable potassium values (source data errors)
    if "Potassium (mg)" in df.columns:
        max_potassium = 5000
        bad_k = (df["Potassium (mg)"] > max_potassium) & df["Potassium (mg)"].notna()
        if bad_k.any():
            print(
                f"Setting {bad_k.sum()} nectar products with Potassium > {max_potassium}mg to NaN"
            )
            df.loc[bad_k, "Potassium (mg)"] = np.nan

    df = _add_product_type_columns(df)
    return df


def filter_for_ranking(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only meat-free in meat categories and dairy-free in dairy categories."""
    mask = ((df["product_type"] == "meat-free") & (df["is_meat_category"] == True)) | (
        (df["product_type"] == "dairy-free") & (df["is_dairy_category"] == True)
    )
    return df[mask].copy()

def get_non_analog_keys(labels_path: str) -> set:
    """Return set of (category, product_code) for non-analog NECTAR products.

    Non-analog = has_meat in meat categories or has_dairy in dairy categories.
    """
    labels = pd.read_csv(labels_path)
    non_analog = (
        ((labels["category"].isin(MEAT_CATEGORIES)) & (labels["has_meat"] == True))
        | ((labels["category"].isin(DAIRY_CATEGORIES)) & (labels["has_dairy"] == True))
    )
    return set(
        zip(labels.loc[non_analog, "category"], labels.loc[non_analog, "product_code"])
    )


def check_taste_like_plant_based(taste_like_df: pd.DataFrame) -> None:
    """Warn if any Taste Like products contain obvious animal ingredient keywords."""
    animal_kw = {
        "beef", "pork", "chicken", "turkey", "lamb", "fish", "salmon",
        "shrimp", "milk", "cream", "butter", "cheese", "whey",
        "casein", "lactose", "gelatin", "lard", "tallow",
    }
    flagged = []
    for _, row in taste_like_df.iterrows():
        ingredients = str(row.get("Ingredient List", "")).lower()
        tokens = {t.strip() for t in ingredients.split(",")}
        found = tokens & animal_kw
        if found:
            flagged.append((row["Category"], row["Product_Code"], found))
    if flagged:
        print(f"WARNING: {len(flagged)} Taste Like products contain animal keywords:")
        for cat, code, kws in flagged[:10]:
            print(f"  {cat} {code}: {kws}")
    else:
        print("All Taste Like products pass animal-keyword check")


In [ ]:
def load_taste_like_data(products_path: str, parents_path: str) -> pd.DataFrame:
    products = pd.read_csv(products_path)
    parents = pd.read_csv(parents_path)

    # Map parent slug -> nectar subcategory
    products["Category"] = products["Product"].map(PARENT_SLUG_TO_NECTAR_CATEGORY)
    products = products[products["Category"].notna()].copy()

    # Drop excluded categories
    products = products[~products["Category"].isin(DROP_CATEGORIES)].copy()

    # Rename and select nutrition columns
    products.rename(
        columns={
            "Ingredients": "Ingredient List",
            "Serving size": "Serving Size (g)",
            "Fat": "Total Fat (g)",
            "Carbs": "Total Carbohydrate (g)",
            "Cholesterol": "Cholesterol (mg)",
            "Sodium": "Sodium (mg)",
            "Protein": "Protein (g)",
            "Calcium": "Calcium (mg)",
            "Iron": "Iron (mg)",
        },
        inplace=True,
    )

    # Strip brand prefixes from ingredients (e.g. "Gardein: Water, ..." -> "Water, ...")
    products["Ingredient List"] = products["Ingredient List"].str.replace(
        r"^[A-Za-z][A-Za-z\s''.&®™️()-]*?:\s*", "", regex=True
    )

    # Convert ml serving sizes to grams (approximate: density ~1.03 for milk/liquids)
    ml_mask = products["Serving size unit"] == "milliliters"
    products.loc[ml_mask, "Serving Size (g)"] = (
        products.loc[ml_mask, "Serving Size (g)"] * 1.03
    ).round(1)

    # Cap unreasonable potassium values (source data errors)
    max_potassium = 5000  # mg per serving
    bad_k = (
        (products["Potassium (mg)"] > max_potassium)
        & products["Potassium (mg)"].notna()
        if "Potassium (mg)" in products.columns
        else pd.Series(False, index=products.index)
    )
    if bad_k.any():
        print(
            f"Setting {bad_k.sum()} taste_like products with Potassium > {max_potassium}mg to NaN"
        )
        products.loc[bad_k, "Potassium (mg)"] = np.nan

    # Cap unreasonable serving sizes (source data errors, e.g. 236588 ml)
    max_serving = 500  # grams; largest nectar serving is 480g
    bad_serving = products["Serving Size (g)"] > max_serving
    if bad_serving.any():
        print(
            f"Capping {bad_serving.sum()} taste_like products with serving size > {max_serving}g"
        )
        products.loc[bad_serving, "Serving Size (g)"] = np.nan

    # Round all nutrition values to 1 decimal place to match nectar precision
    nutr_cols_present = [c for c in NUTRITION_COLS if c in products.columns]
    for col in nutr_cols_present:
        products[col] = products[col].round(1)

    # Round Calories and Cholesterol to integers (nectar always reports whole numbers)
    for int_col in ["Calories", "Cholesterol (mg)"]:
        if int_col in products.columns:
            products[int_col] = products[int_col].round(0)

    # Assign sequential product codes (unique per product variant)
    products = products.reset_index(drop=True)
    products["Product_Code"] = products.index + 1

    products["source"] = "taste_like"
    products["Year"] = None

    # Drop rows missing ingredients or calories (unusable)
    products = products.dropna(subset=["Ingredient List", "Calories"])

    products = _add_product_type_columns(products)
    return products

In [5]:
import re


def normalize_ingredients(text: str) -> set:
    """Normalize an ingredient string into a set of lowercase ingredient tokens."""
    if pd.isna(text):
        return set()
    text = str(text)
    # Remove parenthetical sub-ingredients
    text = re.sub(r"\([^)]*\)", "", text)
    # Split by comma, lowercase, strip whitespace
    items = [item.strip().lower() for item in text.split(",")]
    # Remove empty strings and very short tokens
    return {item for item in items if len(item) > 1}


def deduplicate_taste_like(
    nectar_df: pd.DataFrame, taste_like_df: pd.DataFrame, threshold: float = 0.8
) -> pd.DataFrame:
    """Remove taste_like products that are near-duplicates of nectar products.

    Uses Jaccard similarity on normalized ingredient sets, compared within
    the same category. Returns the filtered taste_like DataFrame.
    """
    taste_like_df = taste_like_df.copy()

    # Build nectar ingredient sets per category
    nectar_by_cat = {}
    for _, row in nectar_df.iterrows():
        cat = row["Category"]
        ing = row.get("Ingredient List") or row.get("Clean Ingredient List")
        ing_set = normalize_ingredients(ing)
        if ing_set:
            nectar_by_cat.setdefault(cat, []).append((row["Product_Code"], ing_set))

    # Check each taste_like product against nectar products in the same category
    drop_indices = []
    matches = []
    for idx, row in taste_like_df.iterrows():
        cat = row["Category"]
        tl_set = normalize_ingredients(row["Ingredient List"])
        if not tl_set or cat not in nectar_by_cat:
            continue
        for nectar_code, nectar_set in nectar_by_cat[cat]:
            intersection = len(tl_set & nectar_set)
            union = len(tl_set | nectar_set)
            if union == 0:
                continue
            jaccard = intersection / union
            if jaccard >= threshold:
                drop_indices.append(idx)
                matches.append(
                    {
                        "category": cat,
                        "taste_like_code": row["Product_Code"],
                        "nectar_code": nectar_code,
                        "jaccard": round(jaccard, 3),
                    }
                )
                break  # one match is enough to remove

    taste_like_df = taste_like_df.drop(index=drop_indices)
    print(
        f"Removed {len(drop_indices)} duplicate taste_like products (Jaccard >= {threshold})"
    )
    if matches:
        display(pd.DataFrame(matches))
    return taste_like_df


def cap_taste_like(
    nectar_ranking_df: pd.DataFrame,
    taste_like_df: pd.DataFrame,
    rng: np.random.RandomState,
    cap_ratio: int = 1,
) -> pd.DataFrame:
    """Cap taste_like products to at most cap_ratio * nectar ranking count per category."""
    nectar_counts = nectar_ranking_df.groupby("Category").size()
    capped_parts = []
    for cat, group in taste_like_df.groupby("Category"):
        max_count = nectar_counts.get(cat, 0) * cap_ratio
        if len(group) > max_count and max_count > 0:
            capped_parts.append(group.sample(n=max_count, random_state=rng))
        else:
            capped_parts.append(group)
    result = pd.concat(capped_parts, ignore_index=True)
    print(f"Capped taste_like: {len(taste_like_df)} -> {len(result)}")
    return result

In [ ]:
def impute_missing_nutrition(
    nectar_df: pd.DataFrame, taste_like_df: pd.DataFrame, rng: np.random.RandomState
) -> pd.DataFrame:
    """Fill missing nutrition columns in taste_like to match nectar distributions and NaN patterns.

    Uses zero-inflated sampling: for each missing value, first decide "zero or nonzero"
    at the nectar per-category zero rate, then if nonzero sample from
    normal(nonzero_mean, nonzero_std). This matches the zero-heavy distributions
    common in food label data (e.g., 82% of PUFA values are 0.0 in nectar).

    NaN is reintroduced using correlated group patterns discovered from nectar
    (PUFA/MUFA always missing together, VitD/Ca/Fe/K always missing together).
    """
    taste_like_df = taste_like_df.copy()

    all_cols = NUTRITION_COLS

    # --- Correlated NaN groups from nectar ---
    NAN_GROUPS = [
        ["Polyunsaturated Fat (g)", "Monounsaturated Fat (g)"],
        ["Vitamin D (mcg)", "Calcium (mg)", "Iron (mg)", "Potassium (mg)"],
    ]
    grouped_cols = {c for g in NAN_GROUPS for c in g}
    independent_cols = [c for c in all_cols if c not in grouped_cols]

    # --- Compute per-category zero-inflated stats ---
    cat_zero_rates = {}
    cat_nonzero_stats = {}
    global_zero_rates = {}
    global_nonzero_stats = {}
    for col in all_cols:
        if col not in nectar_df.columns:
            continue
        # Per-category
        zero_rates = {}
        nz_stats = {}
        for cat, group in nectar_df.groupby("Category"):
            vals = group[col].dropna()
            if len(vals) == 0:
                continue
            zero_rates[cat] = (vals == 0).mean()
            nonzero = vals[vals != 0]
            if len(nonzero) > 0:
                nz_stats[cat] = {"mean": nonzero.mean(), "std": nonzero.std()}
            else:
                nz_stats[cat] = {"mean": 0, "std": 0}
        cat_zero_rates[col] = zero_rates
        cat_nonzero_stats[col] = nz_stats
        # Global
        all_vals = nectar_df[col].dropna()
        global_zero_rates[col] = (all_vals == 0).mean() if len(all_vals) > 0 else 0
        nonzero_all = all_vals[all_vals != 0]
        if len(nonzero_all) > 0:
            global_nonzero_stats[col] = {
                "mean": nonzero_all.mean(),
                "std": nonzero_all.std(),
            }
        else:
            global_nonzero_stats[col] = {"mean": 0, "std": 0}

    # --- Compute per-category group NaN rates ---
    group_nan_rates = {}
    global_group_nan_rates = {}
    for gi, group in enumerate(NAN_GROUPS):
        present = [c for c in group if c in nectar_df.columns]
        if not present:
            continue
        all_nan = nectar_df[present].isna().all(axis=1)
        rates = all_nan.groupby(nectar_df["Category"]).mean()
        group_nan_rates[gi] = rates
        global_group_nan_rates[gi] = all_nan.mean()

    indep_nan_rates = {}
    global_indep_nan_rates = {}
    for col in independent_cols:
        if col not in nectar_df.columns:
            continue
        rates = nectar_df.groupby("Category")[col].apply(lambda s: s.isna().mean())
        indep_nan_rates[col] = rates
        global_indep_nan_rates[col] = nectar_df[col].isna().mean()

    # --- Step 1: Fill missing values using zero-inflated sampling ---
    for col in all_cols:
        if col not in nectar_df.columns:
            continue
        if col not in taste_like_df.columns:
            taste_like_df[col] = np.nan

        values = []
        for _, row in taste_like_df.iterrows():
            cat = row["Category"]
            existing = row.get(col)
            if pd.notna(existing):
                values.append(existing)
            else:
                # Get zero rate for this category
                zero_rate = cat_zero_rates.get(col, {}).get(
                    cat, global_zero_rates.get(col, 0)
                )

                # Zero-inflated draw: first decide zero or nonzero
                if rng.rand() < zero_rate:
                    values.append(0.0)
                else:
                    # Sample from nonzero distribution
                    nz = cat_nonzero_stats.get(col, {}).get(
                        cat, global_nonzero_stats.get(col, {"mean": 0, "std": 0})
                    )
                    mean = nz["mean"]
                    std = nz["std"]
                    if pd.isna(mean) or mean == 0:
                        values.append(0.0)
                        continue
                    if pd.isna(std) or std == 0:
                        std = max(abs(mean) * 0.1, 0.01)
                    val = rng.normal(mean, std)
                    val = max(0, val)
                    values.append(val)
        taste_like_df[col] = values

    # --- Step 2: Reintroduce NaN using correlated groups ---
    for gi, group in enumerate(NAN_GROUPS):
        present = [c for c in group if c in taste_like_df.columns]
        if not present or gi not in group_nan_rates:
            continue
        for idx, row in taste_like_df.iterrows():
            cat = row["Category"]
            rate = group_nan_rates[gi].get(cat, global_group_nan_rates[gi])
            if rng.rand() < rate:
                for c in present:
                    taste_like_df.at[idx, c] = np.nan

    for col in independent_cols:
        if col not in taste_like_df.columns or col not in indep_nan_rates:
            continue
        for idx, row in taste_like_df.iterrows():
            cat = row["Category"]
            rate = indep_nan_rates[col].get(cat, global_indep_nan_rates[col])
            if rng.rand() < rate:
                taste_like_df.at[idx, col] = np.nan

    # --- Step 3: Round to match nectar precision (1 decimal place) ---
    for col in all_cols:
        if col in taste_like_df.columns:
            taste_like_df[col] = taste_like_df[col].round(1)

    # Round Calories and Cholesterol to integers (nectar always reports whole numbers)
    for int_col in ["Calories", "Cholesterol (mg)"]:
        if int_col in taste_like_df.columns:
            taste_like_df[int_col] = taste_like_df[int_col].round(0)

    return taste_like_df

In [7]:
def build_products_df(
    nectar_df: pd.DataFrame, taste_like_df: pd.DataFrame
) -> pd.DataFrame:
    keep_cols = (
        ["Product_Code", "Category", "Ingredient List"]
        + NUTRITION_COLS
        + ["source", "Year"]
    )

    nectar_sub = nectar_df[[c for c in keep_cols if c in nectar_df.columns]].copy()
    taste_like_sub = taste_like_df[
        [c for c in keep_cols if c in taste_like_df.columns]
    ].copy()

    combined = pd.concat([nectar_sub, taste_like_sub], ignore_index=True)
    return combined

In [8]:
def remap_product_codes(products_df: pd.DataFrame, rng: np.random.RandomState) -> tuple:
    products_df = products_df.copy()

    keys = (
        products_df[["Category", "Product_Code", "source"]]
        .drop_duplicates()
        .sort_values(["Category", "Product_Code", "source"])
        .reset_index(drop=True)
    )

    n = len(keys)
    new_codes = rng.permutation(np.arange(100, 100 + n))

    code_map = pd.DataFrame(
        {
            "Category": keys["Category"],
            "Original_Product_Code": keys["Product_Code"],
            "New_Product_Code": new_codes,
            "Source": keys["source"],
        }
    )

    # Build lookup dict
    lookup = {}
    for _, row in code_map.iterrows():
        lookup[(row["Category"], row["Original_Product_Code"], row["Source"])] = row[
            "New_Product_Code"
        ]

    products_df["Product_Code"] = products_df.apply(
        lambda r: lookup[(r["Category"], r["Product_Code"], r["source"])],
        axis=1,
    )

    return products_df, code_map

In [9]:
def compute_mean_similarity(sensory_path: str) -> pd.DataFrame:
    df = pd.read_csv(sensory_path)
    df = df[df["similarity"].notna()].copy()
    mean_sim = (
        df.groupby(["product_category", "product_code"])["similarity"]
        .mean()
        .reset_index()
    )
    mean_sim.columns = ["Category", "Product_Code", "mean_similarity"]
    return mean_sim

In [10]:
def generate_ranking_pairs(
    ranking_products_df: pd.DataFrame, rng: np.random.RandomState
) -> pd.DataFrame:
    """Generate pairs only from the filtered (meat-free / dairy-free) subset."""
    all_pairs = []
    for cat, group in ranking_products_df.groupby("Category"):
        codes = sorted(group["Product_Code"].unique())
        for c1, c2 in combinations(codes, 2):
            all_pairs.append(
                {"product_category": cat, "product_code_1": c1, "product_code_2": c2}
            )

    pairs_df = pd.DataFrame(all_pairs)

    # Randomly swap 50% of pairs
    swap_mask = rng.rand(len(pairs_df)) < 0.5
    pairs_df.loc[swap_mask, ["product_code_1", "product_code_2"]] = pairs_df.loc[
        swap_mask, ["product_code_2", "product_code_1"]
    ].values

    pairs_df.insert(0, "test_id", range(len(pairs_df)))
    return pairs_df

In [11]:
def generate_solution(
    pairs_df: pd.DataFrame,
    mean_sim: pd.DataFrame,
    code_map: pd.DataFrame,
    nectar_df: pd.DataFrame,
) -> tuple:
    # Build reverse lookup: new_code -> (Category, Original_Code, Source)
    reverse_map = {}
    for _, row in code_map.iterrows():
        reverse_map[row["New_Product_Code"]] = (
            row["Category"],
            row["Original_Product_Code"],
            row["Source"],
        )

    # Build set of nectar products with similarity data (2025 and 2026)
    scored_products = set()
    for _, row in nectar_df.iterrows():
        if row["Year"] in (2025, 2026):
            scored_products.add((row["Category"], row["Product_Code"]))

    # Build similarity lookup
    sim_lookup = {}
    for _, row in mean_sim.iterrows():
        sim_lookup[(row["Category"], row["Product_Code"])] = row["mean_similarity"]

    labels = []
    for _, pair in pairs_df.iterrows():
        code1 = pair["product_code_1"]
        code2 = pair["product_code_2"]

        cat1, orig1, src1 = reverse_map[code1]
        cat2, orig2, src2 = reverse_map[code2]

        key1 = (cat1, orig1)
        key2 = (cat2, orig2)

        scored1 = src1 == "nectar" and key1 in scored_products
        scored2 = src2 == "nectar" and key2 in scored_products

        if scored1 and scored2 and key1 in sim_lookup and key2 in sim_lookup:
            sim1 = sim_lookup[key1]
            sim2 = sim_lookup[key2]
            if sim1 > sim2:
                labels.append(code1)
            elif sim2 > sim1:
                labels.append(code2)
            else:
                labels.append(-1)
        else:
            labels.append(-1)

    solution_df = pd.DataFrame(
        {"test_id": pairs_df["test_id"], "higher_rated_product": labels}
    )

    sample_sub_df = pd.DataFrame(
        {
            "test_id": pairs_df["test_id"],
            "higher_rated_product": pairs_df["product_code_1"],
        }
    )

    return solution_df, sample_sub_df

## Step 1 — Load Data

In [12]:
SEED = 42
rng_cap = np.random.RandomState(SEED)
rng_impute = np.random.RandomState(SEED)
rng_remap = np.random.RandomState(SEED)
rng_pairs = np.random.RandomState(SEED)
rng_split = np.random.RandomState(SEED)

sensory_path = (
    base_dir / "data/consolidated_datasets/nectar_consolidated_sensory_rating.csv"
)

nectar_df = load_nectar_data(
    base_dir
    / "data/consolidated_datasets/nectar_consolidated_ingredients_nutrition.csv",
    sensory_path,
)
taste_like_df = load_taste_like_data(
    base_dir / "data/taste_like/Products-Grid view.csv",
    base_dir / "data/taste_like/Product Parent-Grid view.csv",
)

print(f"Nectar products (all): {len(nectar_df)}")
print(f"Taste_like products (raw): {len(taste_like_df)}")

# Remove taste_like products that duplicate nectar products
taste_like_df = deduplicate_taste_like(nectar_df, taste_like_df)

# Cap taste_like to 1x nectar count per category (using ALL nectar as baseline)
taste_like_df = cap_taste_like(nectar_df, taste_like_df, rng_cap)

# Exclude non-analog NECTAR products (animal/hybrid) from ranking pairs.
# These stay in products.csv as reference anchors but are not ranked.
labels_path = base_dir / "shared/data/nectar_product_labels.csv"
non_analog_keys = get_non_analog_keys(labels_path)
nectar_ranking_df = nectar_df[
    ~nectar_df.apply(
        lambda r: (r["Category"], r["Product_Code"]) in non_analog_keys, axis=1
    )
].copy()

print(f"Nectar products (ranking, analog-only): {len(nectar_ranking_df)}")

# Defensive check: all Taste Like products should be plant-based
check_taste_like_plant_based(taste_like_df)

print(f"\nTaste_like products (final): {len(taste_like_df)}")
print(f"\nTaste_like by category:")
print(taste_like_df["Category"].value_counts().sort_index().to_string())


Capping 4 taste_like products with serving size > 500g
Nectar products (all): 247
Nectar products (ranking subset): 247
Taste_like products (raw): 1024
Removed 26 duplicate taste_like products (Jaccard >= 0.8)


,category,taste_like_code,nectar_code,jaccard
0,Meatballs,14,491,1.000
1,Butter,73,334,0.857
2,Bacon,91,334,0.825
3,Hot_Dogs,101,922,1.000
4,Hot_Dogs,114,206,0.905
5,Chicken_Strips,205,702,1.000
6,Chicken_Strips,206,702,0.828
7,Deli_Ham,209,548,0.842
8,Ice_Cream_Hard_Serve,241,491,0.818
9,Ice_Cream_Hard_Serve,271,548,0.867


Capped taste_like: 998 -> 210

Taste_like products (final): 210

Taste_like by category:
Category
Bacon                        9
Barista_Milk                10
Bratwurst                    3
Breaded_Chicken_Filet        6
Breakfast_Sausages          11
Burgers                     11
Butter                      11
Cheddar_Cheese               9
Chicken_Strips              11
Cream_Cheese                11
Creamer                     11
Deli_Ham                    10
Deli_Turkey                  1
Hot_Dogs                     6
Ice_Cream_Hard_Serve        11
Meatballs                    6
Milk                        21
Mozzarella                  11
Nuggets                     11
Pulled_Pork                  6
Sour_Cream                   4
Steak                       10
Unbreaded_Chicken_Breast     2
Yogurt                       8


## Step 2 — Impute Missing Nutrition & Combine

In [13]:
taste_like_df = impute_missing_nutrition(nectar_df, taste_like_df, rng_impute)

products_df = build_products_df(nectar_df, taste_like_df)
print(f"Combined products (all): {len(products_df)}")
print(f"Product types in nectar: {nectar_df['product_type'].value_counts().to_dict()}")
display(products_df[["Category", "Product_Code", "source"] + NUTRITION_COLS[:5]].head())

Combined products (all): 457
Product types in nectar: {'meat-free': 135, 'dairy-free': 112}


,Category,Product_Code,source,Serving Size (g),Calories,Total Fat (g),Saturated Fat (g),Trans Fat (g)
0,Bacon,182,nectar,27.0,180.0,19.0,4.0,0.0
1,Bacon,257,nectar,100.0,218.0,15.0,1.3,0.0
2,Bacon,334,nectar,16.0,60.0,4.5,0.5,0.0
3,Bacon,396,nectar,100.0,181.0,9.6,1.5,0.1
4,Bacon,413,nectar,18.0,60.0,3.0,2.0,0.0


## Step 3 — Remap Product Codes

In [14]:
products_df, code_map = remap_product_codes(products_df, rng_remap)

print(f"Unique remapped codes: {products_df['Product_Code'].nunique()}")
display(code_map.head(10))

Unique remapped codes: 457


,Category,Original_Product_Code,New_Product_Code,Source
0,Bacon,125,404,taste_like
1,Bacon,182,139,nectar
2,Bacon,257,441,nectar
3,Bacon,334,318,nectar
4,Bacon,396,255,nectar
5,Bacon,412,299,taste_like
6,Bacon,413,272,nectar
7,Bacon,491,201,nectar
8,Bacon,513,503,taste_like
9,Bacon,548,130,nectar


## Step 4 — Compute Similarity & Generate Ranking Pairs

In [15]:
mean_sim = compute_mean_similarity(sensory_path)
print(f"Products with similarity scores: {len(mean_sim)}")

# Build ranking subset with remapped codes
ranking_codes = set()
for _, row in nectar_ranking_df.iterrows():
    ranking_codes.add((row["Category"], row["Product_Code"], "nectar"))
for _, row in taste_like_df.iterrows():
    ranking_codes.add((row["Category"], row["Product_Code"], "taste_like"))

# Look up remapped codes for the ranking subset
ranking_lookup = {}
for _, row in code_map.iterrows():
    key = (row["Category"], row["Original_Product_Code"], row["Source"])
    if key in ranking_codes:
        ranking_lookup[key] = row["New_Product_Code"]

ranking_products_df = products_df[
    products_df["Product_Code"].isin(ranking_lookup.values())
].copy()

pairs_df = generate_ranking_pairs(ranking_products_df, rng_pairs)
print(f"Products in ranking subset: {len(ranking_products_df)}")
print(f"Total ranking pairs: {len(pairs_df)}")
display(pairs_df.head())

Products with similarity scores: 249
Products in ranking subset: 457
Total ranking pairs: 4626


,test_id,product_category,product_code_1,product_code_2
0,0,Bacon,139,130
1,1,Bacon,130,155
2,2,Bacon,130,172
3,3,Bacon,130,182
4,4,Bacon,201,130


## Step 5 — Generate Solution

In [16]:
solution_df, sample_sub_df = generate_solution(pairs_df, mean_sim, code_map, nectar_df)
scored = (solution_df["higher_rated_product"] != -1).sum()
print(f"Scored pairs: {scored} / {len(solution_df)}")
display(solution_df[solution_df["higher_rated_product"] != -1].head())

Scored pairs: 1241 / 4626


,test_id,higher_rated_product
0,0,139
3,3,130
4,4,201
6,6,255
8,8,130


## Step 5b — Assign Public / Private Split

Split scored pairs into Public (50%) and Private (50%) for Kaggle leaderboard evaluation.

- Scored pairs are randomly split 50/50, stratified by category.
- Unscored pairs are assigned to Private (they don't affect scoring).

In [17]:
# Identify scored pairs and split 50/50 by category
scored_mask = solution_df["higher_rated_product"] != -1
scored_indices = solution_df[scored_mask].index.tolist()

# Get category for each scored pair
scored_categories = pairs_df.loc[scored_indices, "product_category"]

# Stratified 50/50 split within each category
public_indices = set()
for cat, group_idx in scored_categories.groupby(scored_categories):
    idx_list = group_idx.index.tolist()
    rng_split.shuffle(idx_list)
    mid = len(idx_list) // 2
    public_indices.update(idx_list[:mid])

# Assign Usage column: Public/Private for scored pairs, Ignored for unscored
solution_df["Usage"] = "Ignored"
solution_df.loc[list(public_indices), "Usage"] = "Public"
private_indices = set(scored_indices) - public_indices
solution_df.loc[list(private_indices), "Usage"] = "Private"

public_scored = ((solution_df["Usage"] == "Public") & scored_mask).sum()
private_scored = ((solution_df["Usage"] == "Private") & scored_mask).sum()
ignored = (solution_df["Usage"] == "Ignored").sum()
print(f"Scored pairs: {public_scored + private_scored}")
print(f"  Public: {public_scored}")
print(f"  Private: {private_scored}")
print(f"Ignored (unscored): {ignored}")

Scored pairs: 1241
  Public: 612
  Private: 629
Ignored (unscored): 3385


## Step 6 — Save Outputs

In [ ]:
final_products = products_df.drop(columns=["source", "Year"], errors="ignore")
# Drop any columns that could leak source info
drop_cols = [
    "Clean Ingredient List",
    "join_key",
    "UPC",
    "Brand",
    "Servings Per Container (#)",
    "has_meat",
    "has_dairy",
    "has_egg",
    "is_dairy_category",
    "is_meat_category",
    "has_plant_protein",
    "product_type",
    "missing_sensory",
]
final_products = final_products.drop(
    columns=[c for c in drop_cols if c in final_products.columns]
)

# Strip trademark/registered symbols from ingredient lists (leak brand names)
final_products["Ingredient List"] = (
    final_products["Ingredient List"]
    .str.replace("™", "", regex=False)
    .str.replace("®", "", regex=False)
    .str.replace("©", "", regex=False)
    .str.replace("  ", " ", regex=False)  # clean up double spaces left behind
)


# Normalize all-caps ingredient lists to Title Case (6 nectar products are all-caps,
# 0 taste_like — this is a detectable source signal)
def _normalize_caps(text):
    if pd.isna(text):
        return text
    if text == text.upper() and len(text) > 20:
        return text.title()
    return text


final_products["Ingredient List"] = final_products["Ingredient List"].apply(
    _normalize_caps
)

final_products.rename(columns={"Product_Code": "Product code"}, inplace=True)

# Sort by Category and Product code for clean presentation
final_products = final_products.sort_values(["Category", "Product code"]).reset_index(
    drop=True
)

final_products.to_csv(output_dir / "products.csv", index=False)

code_map.to_csv(output_dir / "product_code_map.csv", index=False)
pairs_df.to_csv(output_dir / "ranking_pairs.csv", index=False)
solution_df.to_csv(output_dir / "solution.csv", index=False)
sample_sub_df.to_csv(output_dir / "sample_submission.csv", index=False)

print(f"Datasets saved to {output_dir.resolve()}")
print(f"Total products: {len(final_products)}")
print(f"  Nectar: {(products_df['source'] == 'nectar').sum()}")
print(f"  Taste_like: {(products_df['source'] == 'taste_like').sum()}")
print(f"Categories: {sorted(final_products['Category'].unique())}")
print(f"Columns: {list(final_products.columns)}")
display(final_products.head())

## Summary by Category

In [19]:
# Products by source and category
product_summary = (
    products_df.groupby(["Category", "source"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={"nectar": "Nectar Products", "taste_like": "Taste Like Products"})
)
product_summary["Total Products"] = product_summary.sum(axis=1)

# Ranking pairs per category
pair_counts = pairs_df.groupby("product_category").size().rename("Ranking Pairs")

# Scored pairs per category (public + private)
scored_pairs = pairs_df.loc[scored_mask[scored_mask].index]
scored_counts = scored_pairs.groupby("product_category").size().rename("Scored Pairs")

# Public/private scored pairs per category
public_scored_mask = (solution_df["Usage"] == "Public") & scored_mask
private_scored_mask = (solution_df["Usage"] == "Private") & scored_mask
public_counts = (
    pairs_df.loc[public_scored_mask[public_scored_mask].index]
    .groupby("product_category")
    .size()
    .rename("Public")
)
private_counts = (
    pairs_df.loc[private_scored_mask[private_scored_mask].index]
    .groupby("product_category")
    .size()
    .rename("Private")
)

summary = (
    product_summary.join(pair_counts)
    .join(scored_counts)
    .join(public_counts)
    .join(private_counts)
    .fillna(0)
    .astype(int)
)
display(summary)
print(f"\nTotals:")
print(summary.sum().to_string())

,Nectar Products,Taste Like Products,Total Products,Ranking Pairs,Scored Pairs,Public,Private
Category,,,,,,,
Bacon,11,9,20,190,55,27,28
Barista_Milk,11,10,21,210,54,27,27
Bratwurst,10,3,13,78,45,22,23
Breaded_Chicken_Filet,6,6,12,66,15,7,8
Breakfast_Sausages,11,11,22,231,55,27,28
Burgers,11,11,22,231,55,27,28
Butter,11,11,22,231,55,27,28
Cheddar_Cheese,11,9,20,190,54,27,27
Chicken_Strips,11,11,22,231,55,27,28



Totals:
Nectar Products         247
Taste Like Products     210
Total Products          457
Ranking Pairs          4626
Scored Pairs           1241
Public                  612
Private                 629
